In [ ]:
%pip install -U transformers accelerate soundfile huggingface_hub


## Local Inference on GPU 
Model page: https://huggingface.co/ibm-granite/granite-speech-4.1-2b

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/ibm-granite/granite-speech-4.1-2b)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
import gc
import soundfile as sf
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoProcessor, pipeline

model_path = "ibm-granite/granite-speech-4.1-2b"
audio_path = hf_hub_download(repo_id=model_path, filename="multilingual_sample.wav")
pipe = pipeline(
    "automatic-speech-recognition",
    model=model_path,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
processor = AutoProcessor.from_pretrained(model_path)
tokenizer = processor.tokenizer
wav, sr = sf.read(audio_path, dtype="float32")
assert sr == 16000
if wav.ndim > 1:
    wav = wav.mean(axis=1)
chat = [{"role": "user", "content": "<|audio|>transcribe the speech with proper punctuation and capitalization."}]
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
model_inputs = processor(prompt, wav, device="cuda", return_tensors="pt").to("cuda")

with torch.no_grad():
    model_outputs = pipe.model.generate(
        **model_inputs,
        max_new_tokens=64,
        do_sample=False,
        num_beams=1,
    )
num_input_tokens = model_inputs["input_ids"].shape[-1]
new_tokens = model_outputs[0, num_input_tokens:].unsqueeze(0)
pipe_text = tokenizer.batch_decode(new_tokens, add_special_tokens=False, skip_special_tokens=True)
print("Pipeline-loaded STT output =", pipe_text[0])
pipe_parameter_devices = sorted({str(p.device) for p in pipe.model.parameters()})
print("pipe parameter devices =", pipe_parameter_devices)

del pipe
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Load model directly
import soundfile as sf
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

device = "cuda"
processor = AutoProcessor.from_pretrained(model_path)
tokenizer = processor.tokenizer
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_path,
    device_map=device,
    torch_dtype=torch.bfloat16,
).eval()

wav, sr = sf.read(audio_path, dtype="float32")
assert sr == 16000
if wav.ndim > 1:
    wav = wav.mean(axis=1)
chat = [{"role": "user", "content": "<|audio|>transcribe the speech with proper punctuation and capitalization."}]
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
model_inputs = processor(prompt, wav, device=device, return_tensors="pt").to(device)

with torch.no_grad():
    model_outputs = model.generate(
        **model_inputs,
        max_new_tokens=64,
        do_sample=False,
        num_beams=1,
    )
num_input_tokens = model_inputs["input_ids"].shape[-1]
new_tokens = model_outputs[0, num_input_tokens:].unsqueeze(0)
output_text = tokenizer.batch_decode(new_tokens, add_special_tokens=False, skip_special_tokens=True)
print("STT output =", output_text[0])
model_parameter_devices = sorted({str(p.device) for p in model.parameters()})
print("model parameter devices =", model_parameter_devices)
